# CAD-AI Inspection — Kaggle POC

Registration-first, model-assisted workflow. Run the cells in order. The original Grounding DINO + DINOv2 baseline is retained. DINOv3 and NVIDIA LocateAnything are optional experiments. **No result is labeled PRESENT/ABSENT unless separately validated.**

In [ ]:
!nvidia-smi
import os, sys, json, subprocess, pathlib, torch
print('Torch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory/1024**3,1),'GB VRAM')

In [ ]:
# Locate the extracted project folder whether this notebook is opened from Kaggle input or working.
from pathlib import Path
candidates = [Path('/kaggle/working/CAD_AI_Inspection'), *Path('/kaggle/input').glob('**/CAD_AI_Inspection')]
ROOT = next((p for p in candidates if (p/'scripts'/'run_registration_poc.py').exists()), None)
if ROOT is None:
    # If the ZIP itself was added as a Kaggle Dataset, extract it.
    zips = list(Path('/kaggle/input').glob('**/CAD_AI_Inspection.zip'))
    if zips:
        import zipfile
        with zipfile.ZipFile(zips[0]) as z: z.extractall('/kaggle/working')
        ROOT = Path('/kaggle/working/CAD_AI_Inspection')
if ROOT is None: raise FileNotFoundError('Add the project ZIP/folder as a Kaggle Dataset input.')
os.chdir(ROOT); print('Project:', ROOT); print('Sample:', (ROOT/'data/sample/AI_sample_CAD').exists())

In [ ]:
# Install project dependencies. Kaggle usually already has a compatible torch build.
!pip -q install -r requirements.txt
!pip -q install -U "transformers==4.57.1" timm


## 1. Dataset audit

In [ ]:
!python scripts/dataset_audit.py --data data/sample/AI_sample_CAD --output outputs/dataset_audit

## 2. Registration-first pipeline

First try automatic SIFT/AKAZE/ORB feature registration. If the view differs too much, use the manual point-selection cell below. A homography is appropriate for one approximately planar face; do not mix points from front/top/side planes.

In [ ]:
import subprocess, sys
res=subprocess.run([sys.executable,'scripts/run_registration_poc.py','--case','data/sample/AI_sample_CAD','--output','outputs/registration_case1_auto','--method','auto'])
if res.returncode:
    print('Automatic registration was not reliable for this view. Continue to manual correspondence selection below; do not treat failure as a component absence.')

### Optional: manual correspondences (when automatic registration fails)

Run this cell in an interactive notebook. Click at least four corresponding points on the same physical plane in CAD first, then in the real photograph, in exactly the same order. Use well-spaced corners/features. This is especially useful for the front panel.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import json
cad_path='data/sample/AI_sample_CAD/FG_00.png'
real_path='data/sample/AI_sample_CAD/BG_00.png'
def pick_points(path, title, n=4):
    im=Image.open(path).convert('RGB')
    fig,ax=plt.subplots(figsize=(12,8)); ax.imshow(im); ax.set_title(f'{title}: click {n} corresponding points in order')
    pts=plt.ginput(n, timeout=0); plt.close(fig)
    if len(pts)!=n: raise RuntimeError('Expected four clicks')
    return [[float(x),float(y)] for x,y in pts]
cad_pts=pick_points(cad_path,'CAD image',4)
real_pts=pick_points(real_path,'Real image',4)
Path('configs').mkdir(exist_ok=True)
Path('configs/kaggle_manual_alignment.json').write_text(json.dumps({'cad_points':cad_pts,'real_points':real_pts},indent=2))
print('Saved configs/kaggle_manual_alignment.json')

In [ ]:
# Run only after the manual point-selection cell has created the JSON file.
manual=Path('configs/kaggle_manual_alignment.json')
if manual.exists():
    !python scripts/run_registration_poc.py --case data/sample/AI_sample_CAD --manual-points configs/kaggle_manual_alignment.json --output outputs/registration_case1_manual
else:
    print('Manual point file not created; skipping.')

## 3. DINOv3 dense-feature experiment

DINOv3 produces dense patch features. This diagnostic compares patch descriptors between the CAD render and photograph. It is not a detector and its cosine values are not presence probabilities.

In [ ]:
!python scripts/compare_dinov3.py --cad data/sample/AI_sample_CAD/FG_00.png --real data/sample/AI_sample_CAD/BG_00.png --output outputs/dinov3_case1 --device cuda

## 4. NVIDIA LocateAnything experiment (optional; large model)

LocateAnything-3B is a 3B-parameter VLM. Its model card reports A100/H100 testing and substantial memory requirements. Skip this on a T4 if it does not have enough free memory. It is not suitable for a 4 GB GTX 1650. Review the model's non-commercial research license before use.

In [ ]:
if torch.cuda.is_available() and torch.cuda.get_device_properties(0).total_memory >= 24*1024**3:
    !pip -q install "transformers==4.57.1" peft decord
    !python scripts/run_locateanything.py --image data/sample/AI_sample_CAD/BG_00.png --output outputs/locateanything_case1
else:
    print('Skipped: LocateAnything-3B experiment requires a high-memory GPU; detected GPU is below 24 GB VRAM or unavailable.')

## 5. Original Grounding DINO + DINOv2 baseline

Run the original baseline as a separate experiment. Keep its output separate from registration-first results.

In [ ]:
!python scripts/download_models.py --models grounding-dino dinov2
!python scripts/run_inspection.py --case data/sample/AI_sample_CAD --output outputs/original_baseline_case1 --device auto

## 6. Package results

Download the output ZIP and inspect `summary.json`, `component_review.csv`, alignment overlays, DINOv3 diagnostics, LocateAnything raw output (if run), and baseline reports. Manually label component presence per view before calculating precision/recall/F1.

In [ ]:
import shutil
shutil.make_archive('/kaggle/working/CAD_AI_Inspection_outputs','zip',ROOT/'outputs')
print('/kaggle/working/CAD_AI_Inspection_outputs.zip')